# 01 — OGG Data Exploration

## Goal
Build the first analysis-ready dataset for **Kahului Airport (OGG), Maui** by combining historical domestic flight operations with airport weather and severe-weather incident context.

### Primary sources
- **BTS Reporting Carrier On-Time Performance** — individual domestic flights, schedules, actual times, delays, cancellations, diversions, and delay causes.
- **NOAA/NCEI Local Climatological Data (LCD)** — hourly airport weather observations.
- **NOAA Storm Events Database** — severe-weather incident metadata and narratives.

### Unit of analysis
For the baseline model, **one row = one scheduled flight departing from or arriving at OGG**.


## Initial prediction target

We will derive a categorical target:

- `normal`: completed flight with < 15 min arrival delay
- `delay`: completed flight with 15–179 min arrival delay
- `severe_delay`: completed flight with >= 180 min arrival delay
- `cancelled`: cancelled flight

Diversions will initially be retained as a separate operational flag and can later become a fifth class.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('..')
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
INCIDENT_DIR = ROOT / 'data' / 'incidents'

RAW_DIR, PROCESSED_DIR, INCIDENT_DIR


## Expected BTS fields

We will keep the smallest useful subset first:

`FlightDate`, `Reporting_Airline`, `Flight_Number_Reporting_Airline`, `Origin`, `Dest`, `CRSDepTime`, `DepTime`, `DepDelay`, `CRSArrTime`, `ArrTime`, `ArrDelay`, `Cancelled`, `CancellationCode`, `Diverted`, `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay`, `Distance`.


In [ ]:
BTS_KEEP = [
    'FlightDate', 'Reporting_Airline', 'Flight_Number_Reporting_Airline',
    'Origin', 'Dest', 'CRSDepTime', 'DepTime', 'DepDelay',
    'CRSArrTime', 'ArrTime', 'ArrDelay', 'Cancelled', 'CancellationCode',
    'Diverted', 'CarrierDelay', 'WeatherDelay', 'NASDelay',
    'SecurityDelay', 'LateAircraftDelay', 'Distance'
]
BTS_KEEP


## Load downloaded BTS files

Place BTS CSV files in `data/raw/bts/`. We will start with a manageable recent historical window and expand after validating the pipeline.


In [ ]:
bts_dir = RAW_DIR / 'bts'
bts_files = sorted(bts_dir.glob('*.csv'))
print(f'Found {len(bts_files)} BTS files')
bts_files[:5]


In [ ]:
def load_bts_ogg(files):
    frames = []
    for path in files:
        df = pd.read_csv(path, low_memory=False)
        available = [c for c in BTS_KEEP if c in df.columns]
        df = df[available].copy()
        if {'Origin', 'Dest'}.issubset(df.columns):
            df = df[(df['Origin'] == 'OGG') | (df['Dest'] == 'OGG')]
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

flights = load_bts_ogg(bts_files)
flights.shape


In [ ]:
def assign_disruption_class(row):
    if row.get('Cancelled', 0) == 1:
        return 'cancelled'
    delay = row.get('ArrDelay', np.nan)
    if pd.isna(delay):
        return 'unknown'
    if delay < 15:
        return 'normal'
    if delay < 180:
        return 'delay'
    return 'severe_delay'

if not flights.empty:
    flights['disruption_class'] = flights.apply(assign_disruption_class, axis=1)
    display(flights['disruption_class'].value_counts(dropna=False))


## Expected NOAA LCD weather fields

At minimum we want timestamp-aligned measures for temperature, dew point, relative humidity, station pressure, visibility, wind speed/direction/gusts, precipitation, cloud/sky condition, and weather type. Exact LCD column names can vary by product/version, so the next step is schema inspection after downloading the first OGG station file.


In [ ]:
weather_dir = RAW_DIR / 'weather'
weather_files = sorted(weather_dir.glob('*.csv'))
print(f'Found {len(weather_files)} weather files')

if weather_files:
    weather_sample = pd.read_csv(weather_files[0], low_memory=False)
    print(weather_sample.shape)
    display(pd.DataFrame({'column': weather_sample.columns}))


## Join strategy

1. Convert scheduled flight time at OGG into a proper Hawaii-local timestamp.
2. Normalize NOAA weather timestamps.
3. For each flight, match the nearest prior weather observation (or aggregate over windows such as 1h, 3h, and 6h before scheduled departure/arrival).
4. Add daily/event-level severe-weather indicators from NOAA Storm Events.
5. Later add airline-, airport-, congestion-, and inbound-aircraft features.


## Phase 1 validation checklist

- Confirm exact OGG weather station identifier(s)
- Download first BTS sample window
- Download matching NOAA LCD hourly data
- Verify timestamps and Hawaii time-zone handling
- Inspect missingness and cancellation coding
- Build one merged flight-weather table
- Plot disruption rate against wind, visibility, and precipitation

After these checks pass, expand the historical window and begin baseline modeling.
